In [ ]:
import sys

sys.path.append("../src")
from configs.load_config import load_config
from data_processing.load import load_dataset
from models.build import build_model
import pandas as pd
from types import SimpleNamespace
import numpy as np
from numpyro.infer import Predictive
from data_processing.utils import load_checkpoint
import properscoring as ps

In [ ]:
model_cfg = load_config("../configs/model/bnn_gen_model_nowcast_03_weighted_150.yaml")
data_cfg = load_config(model_cfg.data.path)
file_name = "model_mw_gamma_extended_2026-03-27_06-31-31_nowcast_ELSPOT_NO3"

In [14]:
data_cfg.dataset = "forecast"
data_cfg.ensemble = SimpleNamespace(members=None, variance=False)

In [ ]:
data = load_dataset(data_cfg)
checkpoint = load_checkpoint(file_name=file_name)

Loading forecast data for ELSPOT NO3, ensemble members [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14]...
Loaded scaler parameters for ELSPOT NO3 and nowcast
  Saved at: 2026-03-20 10:19:42
  Split time: 2025-01-01 00:00:00+01:00
Dropping 15 gust columns not present in nowcast

Column mapping summary:
  Original columns: 94
  Renamed columns: 79
  Dropped columns: 15
  Final columns: 79

 RangeIndex(start=0, stop=6874212, step=1)
All parks operational from 2022-10-12 12:00:00
Before 2-day ahead filtering: 1316376 rows
After 2-day ahead filtering: 489192 rows
Lead time range: 34.0h to 61.0h


/Users/noemiweber/Documents/Studium/master/4.semester_norway/WindAIcompetition/notebooks/../src/data_processing/downtime.py:64: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  maintenance_events.drop(


Maintenance hours in operational period: 451
Preprocessing done. X_train shape: (18069, 24, 15, 10)
Preprocessing done. X_test shape: (1895, 24, 15, 10)


In [ ]:
def evaluate_sample_sizes(predictions, y_test, sample_sizes):

    results = []

    for n in sample_sizes:
        subset = predictions[:n]  # first n retained samples

        # Posterior predictive mean for point prediction
        pred_mean = subset.mean(axis=0)
        pred_median = np.percentile(subset, 50, axis=0)

        rmse_n = np.sqrt(np.mean((pred_median - y_test) ** 2))  # TODO: mean or median??
        crps_n = ps.crps_ensemble(y_test, subset, axis=0).mean()

        results.append({"n_samples": n, "RMSE": rmse_n, "CRPS": crps_n})

    return pd.DataFrame(results)

In [ ]:
def evaluate_sample_subsets(
    data,
    file_name,
    checkpoint,
    sample_sizes=[
        10,
        20,
        30,
        50,
        100,
    ],
):
    results_dfs = []

    checkpoint = load_checkpoint(file_name=file_name)
    model = build_model(checkpoint["model_cfg"], data)
    samples = checkpoint["samples"]
    predictive = Predictive(
        model,
        samples,
        return_sites=["mu", "alpha", "Y"],
    )

    for member in range(15):

        if model_cfg.backend == "numpyro":

            pp_samples = predictive(
                checkpoint["rng_key_predict"], X=data["X_test"][:, :, member], Y=None
            )
            y_pred = pp_samples["Y"]
            results_df = evaluate_sample_sizes(y_pred, data["Y_test_gen"], sample_sizes)
            results_dfs.append(results_df)

        if data_cfg.dataset == "nowcast":
            # validate on nowcast, only one loop, otherwise validation on all ensemble members
            print("break")
            break

    result = pd.concat(results_dfs).groupby("n_samples", as_index=False).mean()

    return result


evaluate_sample_subsets(data, file_name, checkpoint)

Loading specified checkpoint: ../checkpoints/model_mw_gamma_extended_2026-03-27_06-31-31_nowcast_ELSPOT_NO3.pkl
